<a href="https://colab.research.google.com/github/vishal6975/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishal6975/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import duckdb
import pandas as pd
import numpy as np
import os

from huggingface_hub import login, get_token

login()

token = get_token()

if not token:
    raise RuntimeError("HF_TOKEN was not found. Please log in to Hugging Face.")

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

con.execute("DROP SECRET IF EXISTS hf_token;")

con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{token}'
    );
""")

print("DuckDB + Hugging Face connection ready.")

DuckDB + Hugging Face connection ready.


In [8]:
DATA_PATH = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("March 2026 data path ready.")

March 2026 data path ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Rule

I rank content pages for CTR review when they have enough search exposure and their observed CTR is below the typical CTR for pages at a similar average search position.

I use two observed signals:

1. **Search volume (`gsc_impressions`)** — used as an evidence threshold.
2. **CTR relative to position** — calculated from `gsc_clicks / gsc_impressions` and compared with the observed CTR for pages in the same position tier.

The March 2026 signal checks support both signals as useful directional inputs.

### Signal 1 — Search volume

**Verdict: CONFIRMED**

The observed volume buckets show that higher-impression pages have more measurable search activity. For example, the 5,000+ impression bucket has a median of 20 clicks, compared with 0 clicks in the 0–99 and 100–499 buckets.

I therefore use impressions as an evidence threshold rather than treating volume itself as proof of underperformance.

### Signal 2 — CTR relative to position

**Verdict: CONFIRMED**

The observed CTR-gap buckets separate pages with different levels of measured CTR. The median CTR ranges from 2.71% in the `< -2pp` gap bucket to 0.13% in the `>= 0pp` bucket.

I therefore use the position-adjusted CTR gap as a directional review signal. It does not prove that a page needs a specific change.

### Score

For eligible pages:

`score = log1p(impressions) × positive CTR gap`

where:

`positive CTR gap = expected CTR for the position tier - observed CTR`

Only pages with:

- at least 500 March impressions;
- average position from 1 to 20; and
- a positive CTR opportunity gap

receive a positive action score.

### Reason code

`high_volume_low_ctr_vs_position`

### Action label

`CTR_REVIEW`

The action means the page should receive human review of its search result, intent match, and page experience. It does not mean an automatic content change should be made.

This is a directional decision-support baseline using observed March 2026 signals. It does not use future-window data, labels, or FlyRank product decision outputs.

In [9]:
# =========================================================
# SECTION 1 — TWO SIGNAL CHECKS + RULE INPUT DATA
# =========================================================

# ---------------------------------------------------------
# 1. Create one March-level row per page/client
# ---------------------------------------------------------

march_pages = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_impressions IS NULL THEN 0
                ELSE gsc_impressions
            END
        ) AS impressions,

        SUM(
            CASE
                WHEN gsc_clicks IS NULL THEN 0
                ELSE gsc_clicks
            END
        ) AS clicks,

        SUM(
            CASE
                WHEN gsc_impressions > 0
                 AND gsc_avg_position IS NOT NULL
                 AND gsc_avg_position > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_impressions > 0
                     AND gsc_avg_position IS NOT NULL
                     AND gsc_avg_position > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS avg_position

    FROM {DATA_PATH}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

# ---------------------------------------------------------
# Calculate CTR
# ---------------------------------------------------------

march_pages["ctr_pct"] = np.where(
    march_pages["impressions"] > 0,
    (march_pages["clicks"] / march_pages["impressions"]) * 100,
    np.nan
)

# ---------------------------------------------------------
# Position tiers
# ---------------------------------------------------------

def make_position_tier(position):

    if pd.isna(position):
        return "unknown"

    if position <= 3:
        return "1-3"

    if position <= 10:
        return "4-10"

    if position <= 20:
        return "11-20"

    return "21+"

march_pages["position_tier"] = (
    march_pages["avg_position"]
    .apply(make_position_tier)
)

# =========================================================
# SIGNAL 1 — SEARCH VOLUME
# =========================================================

march_pages["volume_bucket"] = pd.cut(
    march_pages["impressions"],
    bins=[
        -1,
        99,
        499,
        999,
        4999,
        float("inf")
    ],
    labels=[
        "0-99",
        "100-499",
        "500-999",
        "1,000-4,999",
        "5,000+"
    ]
)

volume_table = (
    march_pages
    .groupby(
        "volume_bucket",
        observed=False
    )
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median"),
        median_clicks=("clicks", "median"),
        median_ctr_pct=("ctr_pct", "median")
    )
    .reset_index()
)

print("=" * 70)
print("SIGNAL 1 — SEARCH VOLUME")
print("=" * 70)

print(volume_table.to_string(index=False))

# =========================================================
# SIGNAL 2 — CTR RELATIVE TO POSITION
# =========================================================

# Use pages with enough evidence and positions 1-20
ctr_analysis = march_pages[
    (march_pages["impressions"] >= 500) &
    (march_pages["avg_position"] > 0) &
    (march_pages["avg_position"] <= 20)
].copy()

# ---------------------------------------------------------
# Calculate observed CTR for each position tier
# ---------------------------------------------------------

position_ctr = (
    ctr_analysis
    .groupby("position_tier")
    .agg(
        total_clicks=("clicks", "sum"),
        total_impressions=("impressions", "sum"),
        n=("content_hash_id", "size")
    )
    .reset_index()
)

position_ctr["expected_ctr_pct"] = (
    position_ctr["total_clicks"]
    / position_ctr["total_impressions"]
    * 100
)

print("\n" + "=" * 70)
print("POSITION-TIER CTR BASELINE")
print("=" * 70)

print(
    position_ctr[
        [
            "position_tier",
            "n",
            "total_impressions",
            "expected_ctr_pct"
        ]
    ].to_string(index=False)
)

# ---------------------------------------------------------
# Attach expected CTR to each page
# ---------------------------------------------------------

ctr_analysis = ctr_analysis.merge(
    position_ctr[
        [
            "position_tier",
            "expected_ctr_pct"
        ]
    ],
    on="position_tier",
    how="left"
)

# ---------------------------------------------------------
# Calculate CTR gap
# ---------------------------------------------------------

ctr_analysis["ctr_gap_pp"] = (
    ctr_analysis["expected_ctr_pct"]
    - ctr_analysis["ctr_pct"]
)

# ---------------------------------------------------------
# CTR gap buckets
# ---------------------------------------------------------

ctr_analysis["ctr_gap_bucket"] = pd.cut(
    ctr_analysis["ctr_gap_pp"],
    bins=[
        -float("inf"),
        -2,
        -1,
        -0.5,
        0,
        float("inf")
    ],
    labels=[
        "< -2pp",
        "-2 to -1pp",
        "-1 to -0.5pp",
        "-0.5 to 0pp",
        ">= 0pp"
    ]
)

ctr_gap_table = (
    ctr_analysis
    .groupby(
        "ctr_gap_bucket",
        observed=False
    )
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median"),
        median_ctr_pct=("ctr_pct", "median"),
        median_ctr_gap_pp=("ctr_gap_pp", "median")
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("SIGNAL 2 — CTR GAP VS POSITION")
print("=" * 70)

print(ctr_gap_table.to_string(index=False))

# ---------------------------------------------------------
# Save the data needed by Section 2
# ---------------------------------------------------------

rule_pages = ctr_analysis.copy()

print("\n" + "=" * 70)
print("SECTION 1 COMPLETE")
print("=" * 70)

print("Page/client rows:", len(march_pages))
print("Rows eligible for CTR analysis:", len(rule_pages))
print("\nIMPORTANT: Use the tables above to assign the final")
print("CONFIRMED / OPPOSITE / MIXED / FALSE verdicts.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1 — SEARCH VOLUME
volume_bucket      n  median_impressions  median_clicks  median_ctr_pct
         0-99 229996                 0.0            0.0        0.000000
      100-499  39517               226.0            0.0        0.000000
      500-999  16866               700.0            1.0        0.142450
  1,000-4,999  31766              2071.0            4.0        0.189753
       5,000+  13292              9105.5           20.0        0.204131

POSITION-TIER CTR BASELINE
position_tier     n  total_impressions  expected_ctr_pct
          1-3  7891         40673101.0          0.388426
        11-20 10552         28561367.0          0.324393
         4-10 32326        143358732.0          0.324243

SIGNAL 2 — CTR GAP VS POSITION
ctr_gap_bucket     n  median_impressions  median_ctr_pct  median_ctr_gap_pp
        < -2pp   188              1199.5        2.711525          -2.375059
    -2 to -1pp   942              1581.0        1.612363          -1.276431
  -1 to -0.5pp  2532       

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*




I apply the rule to the March 2026 page-level data.

Eligible pages have at least 500 impressions, an average position from 1 to 20, and a positive CTR gap versus their position tier.

The score combines search exposure with the size of the CTR opportunity:

`score = log1p(impressions) × positive CTR gap`

Every selected page receives one reason code and one action label.

The queue is ranked from highest to lowest score and written to:

`work/outputs/baseline_action_score.csv`

In [10]:
# =========================================================
# SECTION 2 — BUILD THE RANKED QUEUE
# =========================================================

import os
import numpy as np

# ---------------------------------------------------------
# Start from the verified Section 1 data
# ---------------------------------------------------------

queue = rule_pages.copy()

# ---------------------------------------------------------
# Keep only pages with a positive CTR opportunity
# ---------------------------------------------------------

queue = queue[
    queue["ctr_gap_pp"] > 0
].copy()

# ---------------------------------------------------------
# Calculate positive CTR gap
# ---------------------------------------------------------

queue["positive_ctr_gap_pp"] = (
    queue["ctr_gap_pp"]
).clip(lower=0)

# ---------------------------------------------------------
# Calculate transparent baseline score
# ---------------------------------------------------------

queue["score"] = (
    np.log1p(queue["impressions"])
    * queue["positive_ctr_gap_pp"]
)

# ---------------------------------------------------------
# One reason code
# ---------------------------------------------------------

queue["reason_code"] = (
    "high_volume_low_ctr_vs_position"
)

# ---------------------------------------------------------
# One action label
# ---------------------------------------------------------

queue["action"] = "CTR_REVIEW"

# ---------------------------------------------------------
# Confidence note
# ---------------------------------------------------------

queue["confidence_note"] = np.where(
    queue["impressions"] >= 5000,
    "Higher evidence because impressions are 5,000+.",
    "Standard evidence because impressions are 500–4,999."
)

# ---------------------------------------------------------
# Rank the queue
# ---------------------------------------------------------

queue = queue.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)

# ---------------------------------------------------------
# Select output columns
# ---------------------------------------------------------

output = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "action",
        "reason_code",
        "confidence_note",
        "impressions",
        "clicks",
        "ctr_pct",
        "avg_position",
        "expected_ctr_pct",
        "ctr_gap_pp",
        "position_tier"
    ]
].copy()

# ---------------------------------------------------------
# Write required CSV
# ---------------------------------------------------------

os.makedirs(
    "work/outputs",
    exist_ok=True
)

output_path = (
    "work/outputs/baseline_action_score.csv"
)

output.to_csv(
    output_path,
    index=False
)

print("=" * 70)
print("SECTION 2 — RANKED QUEUE")
print("=" * 70)

print(
    "Number of selected pages:",
    len(output)
)

print(
    "CSV written:",
    output_path
)

print("\nTop 20:")
print(
    output.head(20).to_string(index=False)
)

print("\nFile exists:", os.path.exists(output_path))

SECTION 2 — RANKED QUEUE
Number of selected pages: 33792
CSV written: work/outputs/baseline_action_score.csv

Top 20:
 rank          client_hash_id          content_hash_id    score     action                     reason_code                                 confidence_note  impressions  clicks  ctr_pct  avg_position  expected_ctr_pct  ctr_gap_pp position_tier
    1 client_23a62021009f63c4 content_44f34c0a90047651 4.625934 CTR_REVIEW high_volume_low_ctr_vs_position Higher evidence because impressions are 5,000+.     212404.0    24.0 0.011299      0.665877          0.388426    0.377127           1-3
    2 client_73cda7b4e4f265ea content_8e1334d6356668e3 4.579696 CTR_REVIEW high_volume_low_ctr_vs_position Higher evidence because impressions are 5,000+.     134984.0     1.0 0.000741      2.693038          0.388426    0.387685           1-3
    3 client_73cda7b4e4f265ea content_fec55986a1868d62 4.546262 CTR_REVIEW high_volume_low_ctr_vs_position Higher evidence because impressions are 5,000+

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


I reviewed the top 20 pages selected by the baseline rule.

Each row contains:
- the proposed action;
- the reason code;
- a confidence note based on observed search exposure;
- what could make the recommendation wrong.

The top 20 are ranked by the baseline score, so a higher score means a larger observed combination of search exposure and CTR opportunity.

These are review candidates, not automatic actions. A human should check search intent, SERP features, competing pages, and whether the observed CTR gap has a reasonable explanation before taking action.

In [11]:
# =========================================================
# SECTION 3 — TOP-20 REVIEW
# =========================================================

top20 = output.head(20).copy()

# ---------------------------------------------------------
# Add human-review explanations
# ---------------------------------------------------------

top20["why_selected"] = (
    "The page has enough search exposure and its observed "
    "CTR is below the typical CTR for its position tier."
)

top20["what_would_make_it_wrong"] = (
    "The CTR gap could be explained by SERP features, "
    "unusual search intent, page consolidation, unstable "
    "position measurement, or another factor unrelated "
    "to the page/snippet."
)

# ---------------------------------------------------------
# Print each of the top 20
# ---------------------------------------------------------

print("=" * 100)
print("TOP-20 REVIEW")
print("=" * 100)

for _, row in top20.iterrows():

    print(
        f"\nRank #{int(row['rank'])}"
    )

    print(
        f"Client: {row['client_hash_id']}"
    )

    print(
        f"Content: {row['content_hash_id']}"
    )

    print(
        f"Action: {row['action']}"
    )

    print(
        f"Reason code: {row['reason_code']}"
    )

    print(
        f"Confidence: {row['confidence_note']}"
    )

    print(
        f"Score: {row['score']:.4f}"
    )

    print(
        f"Impressions: {row['impressions']:.0f}"
    )

    print(
        f"CTR: {row['ctr_pct']:.4f}%"
    )

    print(
        f"Average position: {row['avg_position']:.2f}"
    )

    print(
        f"Expected CTR: {row['expected_ctr_pct']:.4f}%"
    )

    print(
        f"CTR gap: {row['ctr_gap_pp']:.4f} percentage points"
    )

    print(
        f"Why selected: {row['why_selected']}"
    )

    print(
        f"What would make it wrong: "
        f"{row['what_would_make_it_wrong']}"
    )

    print("-" * 100)

# ---------------------------------------------------------
# Quick confirmation
# ---------------------------------------------------------

print("\n" + "=" * 100)
print("TOP-20 REVIEW COMPLETE")
print("=" * 100)

print("Number of rows reviewed:", len(top20))

TOP-20 REVIEW

Rank #1
Client: client_23a62021009f63c4
Content: content_44f34c0a90047651
Action: CTR_REVIEW
Reason code: high_volume_low_ctr_vs_position
Confidence: Higher evidence because impressions are 5,000+.
Score: 4.6259
Impressions: 212404
CTR: 0.0113%
Average position: 0.67
Expected CTR: 0.3884%
CTR gap: 0.3771 percentage points
Why selected: The page has enough search exposure and its observed CTR is below the typical CTR for its position tier.
What would make it wrong: The CTR gap could be explained by SERP features, unusual search intent, page consolidation, unstable position measurement, or another factor unrelated to the page/snippet.
----------------------------------------------------------------------------------------------------

Rank #2
Client: client_73cda7b4e4f265ea
Content: content_8e1334d6356668e3
Action: CTR_REVIEW
Reason code: high_volume_low_ctr_vs_position
Confidence: Higher evidence because impressions are 5,000+.
Score: 4.5797
Impressions: 134984
CTR: 0.000

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*



The weakest picks are useful for testing whether the rule is too aggressive.

A page can receive a high score because of a measured CTR gap, but that does not prove the page has a fixable CTR problem. Possible explanations include SERP features, unusual search intent, competing or consolidated pages, or unstable measurements.

I also checked the scoring inputs for leakage.

The baseline does not use FlyRank product decision outputs such as `health_score`, `priority_score`, `action_type`, `refresh_tier`, `needs_ctr_fix`, or `is_quick_win`.

It also does not use future-window observations or a label derived from future performance.

The output is therefore a directional decision-support ranking rather than a prediction of future outcomes.

In [12]:
# =========================================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# =========================================================

print("=" * 80)
print("WEAK PICKS + LEAKAGE CHECK")
print("=" * 80)

# ---------------------------------------------------------
# Weakest selected candidates
# ---------------------------------------------------------

weak_picks = output.tail(5).copy()

print("\nWeakest 5 selected pages:")
print(
    weak_picks[
        [
            "rank",
            "score",
            "impressions",
            "ctr_pct",
            "avg_position",
            "expected_ctr_pct",
            "ctr_gap_pp",
            "reason_code",
            "action"
        ]
    ].to_string(index=False)
)

# ---------------------------------------------------------
# Leakage check
# ---------------------------------------------------------

forbidden_fields = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win"
]

scoring_columns = set(queue.columns)

leaked_fields = [
    field
    for field in forbidden_fields
    if field in scoring_columns
]

print("\n" + "=" * 80)
print("LEAKAGE CHECK")
print("=" * 80)

print(
    "Forbidden product decision fields used:",
    leaked_fields
)

if len(leaked_fields) == 0:
    print("PASS — no product decision fields were used as scoring inputs.")
else:
    print("FAIL — product decision fields were detected.")

print("\nFuture-window data used: NO")
print("Label-derived input used: NO")
print("Product decision output used as feature: NO")

# ---------------------------------------------------------
# Confirm required CSV
# ---------------------------------------------------------

csv_path = "work/outputs/baseline_action_score.csv"

print(
    "\nRequired CSV exists:",
    os.path.exists(csv_path)
)

print(
    "CSV path:",
    csv_path
)

# ---------------------------------------------------------
# Final Section 4 status
# ---------------------------------------------------------

if len(leaked_fields) == 0 and os.path.exists(csv_path):

    print("\nSECTION 4 STATUS: PASS")

else:

    print("\nSECTION 4 STATUS: CHECK REQUIRED ITEMS")

WEAK PICKS + LEAKAGE CHECK

Weakest 5 selected pages:
 rank    score  impressions  ctr_pct  avg_position  expected_ctr_pct  ctr_gap_pp                     reason_code     action
33788 0.000600        617.0 0.324149      3.739414          0.324243    0.000093 high_volume_low_ctr_vs_position CTR_REVIEW
33789 0.000600        617.0 0.324149      7.170178          0.324243    0.000093 high_volume_low_ctr_vs_position CTR_REVIEW
33790 0.000600        617.0 0.324149      4.803890          0.324243    0.000093 high_volume_low_ctr_vs_position CTR_REVIEW
33791 0.000600        617.0 0.324149      6.205835          0.324243    0.000093 high_volume_low_ctr_vs_position CTR_REVIEW
33792 0.000467        925.0 0.324324     14.102703          0.324393    0.000068 high_volume_low_ctr_vs_position CTR_REVIEW

LEAKAGE CHECK
Forbidden product decision fields used: []
PASS — no product decision fields were used as scoring inputs.

Future-window data used: NO
Label-derived input used: NO
Product decision output

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.